# Context cleaning — rules in English, compiled

FreshData compiles natural-language expectations into a deterministic,
reviewable `ContextPolicy` — no model, no network, no new dependency.
This notebook walks the flagship ecommerce example end to end.

In [ ]:
import pandas as pd
import freshdata as fd


## A messy ecommerce customer extract

In [ ]:
df = pd.DataFrame({
    "CustomerID": [101, 102, 102, 104, 105],
    "Full Name": ["Asha Rao", "Ben Cotton", "Ben Cotton", "Dana Li", "Ed Fox"],
    "Email Addr": ["asha@example.com", "ben@example.com", "ben@example.com", "not-an-email", None],
    "Mobile": ["9876543210", None, "9123456789", "9111111111", "9000000001"],
    "Age": [25, None, 40, 35, None],
    "Monthly Revenue": [1200.50, 800.00, 800.00, 99999.0, 430.25],
    "City": ["Pune", "Delhi", "Delhi", "Goa", "Pune"],
})
df

## 1. Write the rules as English

In [ ]:
context = """
This is an ecommerce customer dataset.
CustomerID is unique.
Emails must be valid.
Phone numbers are Indian.
Missing Age should be estimated only if confidence >95%.
Never modify revenue values.
"""

## 2. Compile — and inspect what FreshData understood

`fd.compile_context` is deterministic: the same text and schema always
produce the same policy. Unresolved references and unparsed sentences are
surfaced, never guessed at.

In [ ]:
policy = fd.compile_context(context, df=df)
print(policy.summary())

The policy is a plain JSON artifact — commit it, diff it, review it in a PR.

In [ ]:
print(policy.to_json())  # policy.to_json("policy.json") writes the file

## 3. Clean under the policy

Protected columns are added to `preserve_columns`, unique columns become
id columns, and per-column hints flow into the semantic context. The
report records the compile and every unresolved item.

In [ ]:
cleaned, report = fd.clean(df, context=context, return_report=True)
cleaned

In [ ]:
[a.description for a in report.actions if a.step == "context"]

The protected column is byte-identical — "Never modify revenue values." is a hard rule:

In [ ]:
cleaned["monthly_revenue"].equals(df["Monthly Revenue"])

## 4. Validate without mutating

In [ ]:
findings = fd.validate(df, context=context)
for f in findings:
    print(f.severity.upper(), f.rule_name, "—", f.message)

## 5. Strict mode

In CI you usually want gaps to fail loudly. `strict=True` raises
`fd.PolicyError` on any unresolved or unparsed line, before touching data.

In [ ]:
try:
    fd.clean(df, context=context + "\nAlso make everything shiny.", strict=True)
except fd.PolicyError as err:
    print(err)

## 6. Reuse the compiled policy

In [ ]:
cleaned_again = fd.clean(df, policy=policy)  # skips parsing entirely
cleaned_again.equals(cleaned)

## Limitations (Phase 1)

- **Deterministic tier-0 lexicon only** — a fixed set of ~12 intent
  families; anything outside it is surfaced as *unparsed*, by design.
- No model, no embeddings, no network — and therefore no arbitrary NLU.
- `valid_format` / `locale_format` compile to semantic hints; the
  value-level email/phone repair experts and executable repair plans
  arrive in the next phase.
- Unresolved column references are never guessed: fix the sentence or
  pass the real column name.

See `docs/context-policies.md` for the full reference.